In [ ]:
import mlflow
from torchinfo import summary
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets
from torchvision import transforms
import lightning as L
from mlflow_demo.xai_eval import rcap
from mlflow_demo.xai import gradient_methods
from tqdm import tqdm


L.seed_everything(42)
mlflow.set_tracking_uri('http://localhost:15001')
# device = "mps"
device = "cuda"
batch_size = 256

Seed set to 42


['/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py', '--f=/root/.local/share/jupyter/runtime/kernel-v3b1a04be5a070bd26e9a64b6a9b1cbea5a38807c7.json']
Not in flask environment, skip the initialization, please use the 'flask --app ...' command to start the service
['/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py', '--f=/root/.local/share/jupyter/runtime/kernel-v3b1a04be5a070bd26e9a64b6a9b1cbea5a38807c7.json']
Not in flask environment, skip the initialization, please use the 'flask --app ...' command to start the service


### Dataset


In [21]:
ls = glob(data_root + "/train/**/*.png", recursive=True)
ls

['/root/autodl-tmp/ml_data/cifar/cifar10/cifar10/train/airplane/0001.png',
 '/root/autodl-tmp/ml_data/cifar/cifar10/cifar10/train/airplane/0002.png',
 '/root/autodl-tmp/ml_data/cifar/cifar10/cifar10/train/airplane/0003.png',
 '/root/autodl-tmp/ml_data/cifar/cifar10/cifar10/train/airplane/0004.png',
 '/root/autodl-tmp/ml_data/cifar/cifar10/cifar10/train/airplane/0005.png',
 '/root/autodl-tmp/ml_data/cifar/cifar10/cifar10/train/airplane/0006.png',
 '/root/autodl-tmp/ml_data/cifar/cifar10/cifar10/train/airplane/0007.png',
 '/root/autodl-tmp/ml_data/cifar/cifar10/cifar10/train/airplane/0008.png',
 '/root/autodl-tmp/ml_data/cifar/cifar10/cifar10/train/airplane/0009.png',
 '/root/autodl-tmp/ml_data/cifar/cifar10/cifar10/train/airplane/0010.png',
 '/root/autodl-tmp/ml_data/cifar/cifar10/cifar10/train/airplane/0011.png',
 '/root/autodl-tmp/ml_data/cifar/cifar10/cifar10/train/airplane/0012.png',
 '/root/autodl-tmp/ml_data/cifar/cifar10/cifar10/train/airplane/0013.png',
 '/root/autodl-tmp/ml_dat

In [ ]:
user_home = os.path.expanduser('~')
data_root = os.path.join(user_home, 'autodl-tmp',
                         'ml_data', 'cifar', 'cifar10', 'cifar10')

trainsform = transforms.Compose([
    transforms.Resize((32, 32)),
])

from backend_central_dev.utils import data_utils

class MyImageDataset(Dataset):
    def __init__(self, img_dir, train=True, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.train = train

        if self.train:
            self.image_paths = glob(self.img_dir + "/train/**/*.png", recursive=True)
        else:
            self.image_paths = glob(self.img_dir + "/test/**/*.png", recursive=True)
        
        self.labels = [os.path.basename(os.path.dirname(path)) for path in self.image_paths]
        
        self.class_label_map = {
            "airplane": 0,
            "automobile": 1,
            "bird": 2,
            "cat": 3,
            "deer": 4,
            "dog": 5,
            "frog": 6,
            "horse": 7,
            "ship": 8,
            "truck": 9
        }
        
        print(f"Found {len(self.image_paths)} images in {self.img_dir}")
        
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.class_label_map[self.labels[idx]]
        image = data_utils.read_image_to_tensor_from_path(img_path)
        if self.transform:
            image = self.transform(image)
        return image, label


train_dataset = MyImageDataset(
    data_root, train=True, transform=trainsform)

# test_dataset = datasets.ImageFolder(
#     root=os.path.join(data_root, 'test'), transform=trainsform)

# train_dataloader = DataLoader(train_dataset, batch_size=batch_size,
#                               shuffle=True, num_workers=0)
# test_dataloader = DataLoader(test_dataset, batch_size=batch_size,
#                              shuffle=False, num_workers=0)

Found 50000 images in /root/autodl-tmp/ml_data/cifar/cifar10/cifar10


(tensor([[[0.1098, 0.1451, 0.1490,  ..., 0.2980, 0.3176, 0.3333],
          [0.1294, 0.1333, 0.1255,  ..., 0.3725, 0.3765, 0.3333],
          [0.1529, 0.1569, 0.2235,  ..., 0.3647, 0.4196, 0.3725],
          ...,
          [0.3255, 0.3412, 0.3294,  ..., 0.3882, 0.3529, 0.3176],
          [0.3451, 0.3529, 0.3647,  ..., 0.3137, 0.2980, 0.3216],
          [0.3804, 0.3686, 0.3647,  ..., 0.2118, 0.2471, 0.2824]],
 
         [[0.0980, 0.1333, 0.1373,  ..., 0.2627, 0.2824, 0.2980],
          [0.1098, 0.1176, 0.1059,  ..., 0.3216, 0.3216, 0.2824],
          [0.1255, 0.1294, 0.1961,  ..., 0.2980, 0.3490, 0.3020],
          ...,
          [0.2863, 0.3020, 0.2902,  ..., 0.3647, 0.3294, 0.2941],
          [0.2824, 0.2902, 0.3020,  ..., 0.2902, 0.2745, 0.2980],
          [0.3059, 0.2941, 0.2941,  ..., 0.1843, 0.2196, 0.2549]],
 
         [[0.0392, 0.0745, 0.0784,  ..., 0.1529, 0.1686, 0.1843],
          [0.0510, 0.0549, 0.0471,  ..., 0.2157, 0.2196, 0.1765],
          [0.0588, 0.0667, 0.1294,  ...,

### Model


In [ ]:
from torchvision import models
import torchmetrics

class MyLightModel(L.LightningModule):
    def __init__(
        self,
        output_features,
        loss_fn_key="CrossEntropyLoss",
        loss_fn_hparams: dict = {},
        optimizer_key="Adam",
        optimizer_hparams: dict = {},
        task="multiclass",
        weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
    ):
        super().__init__()
        self.save_hyperparameters(ignore=[])

        model = models.efficientnet_v2_s(
            weights=weights
        )
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(
            in_features=in_features, out_features=output_features
        )
        self.model = model

        self.loss_fn = getattr(torch.nn, loss_fn_key)(**loss_fn_hparams)

        self.metrics = dict(
            acc=torchmetrics.Accuracy(
                task=task, top_k=1, num_classes=output_features),
            macro_acc=torchmetrics.Accuracy(
                task=task, top_k=1, average="macro", num_classes=output_features
            ),
            f1=torchmetrics.F1Score(
                task=task, average="macro", num_classes=output_features),
            roc_auc=torchmetrics.AUROC(task=task, num_classes=output_features),
        )

        self.st = None
        self.y_true = np.array([])
        self.y_pred = np.array([])
        self.losses = np.array([])
        self.cm = None
        self.cr = None
        self.metrics_in_batch = {}

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        output = self.model(x)
        loss = self.loss_fn.to(x.device)(output, y)
        self.eval_metrics("train", batch, batch_idx, output, loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        output = self.model(x)
        loss = self.loss_fn.to(x.device)(output, y)
        self.eval_metrics("val", batch, batch_idx, output, loss)

    def configure_optimizers(self):
        params = self.model.parameters()

        self.optimizer = getattr(torch.optim, self.hparams.optimizer_key)(
            params, **self.hparams.optimizer_hparams
        )
        return self.optimizer

    def eval_metrics(self, event_key, batch, batch_idx, output, loss):
        x, y = batch
        metrics = {}
        if self.hparams.task == "multiclass":
            _y = y.detach().cpu()
            odc = output.detach().cpu()
            for k, v in self.metrics.items():
                key = f"{event_key}_{k}"

                # we have to make the y to cpu to avoid MPS glitch
                if len(_y.shape) > 1:
                    _y = _y.argmax(dim=1)
                score = v(odc.float(), _y).item()
                metrics[key] = score
                # self.trainer.progress_bar_metrics[k] = metrics[key]
                if event_key == "val":
                    if self.metrics_in_batch.get(key) is None:
                        self.metrics_in_batch[key] = []
                    self.metrics_in_batch[key].append(score)

            if event_key == "val":
                self.y_true = np.append(self.y_true, _y.numpy())
                self.y_pred = np.append(
                    self.y_pred, torch.argmax(odc, dim=1).numpy()
                )
                self.losses = np.append(self.losses, loss.cpu().item())

                ytp = torch.from_numpy(self.y_pred)
                ytt = torch.from_numpy(self.y_true)
                overall_val_acc = self.metrics["acc"](ytp, ytt)
                self.trainer.progress_bar_metrics["max_val_acc"] = max(
                    overall_val_acc,
                    self.trainer.progress_bar_metrics.get("max_val_acc", 0.0),
                )
                

        metrics[f"{event_key}_loss"] = loss.item()
        self.trainer.progress_bar_metrics["loss"] = metrics[f"{event_key}_loss"]
        self.log_dict(metrics)
        mlflow.log_metrics(metrics, step=self.global_step)


model = MyLightModel(
    10,
    optimizer_hparams=dict(
        lr=0.0001,
        weight_decay=4.0e-05
    )
)
model.hparams

"loss_fn_hparams":   {}
"loss_fn_key":       CrossEntropyLoss
"optimizer_hparams": {'lr': 0.0001, 'weight_decay': 4e-05}
"optimizer_key":     Adam
"output_features":   10
"task":              multiclass
"weights":           EfficientNet_V2_S_Weights.IMAGENET1K_V1

### MLflow


In [8]:
trainer_params = dict(
    max_epochs=40,
    # max_epochs=1,
    # limit_train_batches=1,
    # limit_test_batches=1,
    # limit_val_batches=1,
    callbacks=[
        L.pytorch.callbacks.ModelCheckpoint(
            monitor='val_acc',
            save_top_k=1,
            filename='best-{epoch}-{val_acc:.4f}',
            mode='max',
            save_weights_only=True
        )
    ],
)
trainer = L.Trainer(
    precision='16-mixed',
    **trainer_params
)

with mlflow.start_run():
    L.seed_everything(42)
    run = mlflow.active_run()
    # Log training parameters.
    mlflow.log_params(model.hparams)
    mlflow.log_params({'trainer_params': trainer_params})

    # Log model summary.
    artifact_root_path = os.path.join("runs", run.info.run_id)
    model_summary_path = os.path.join(artifact_root_path, "model_summary.txt")
    os.makedirs(os.path.dirname(model_summary_path), exist_ok=True)
    with open(model_summary_path, "w") as f:
        f.write(str(summary(model)))
    mlflow.log_artifact(model_summary_path)

    print("Start train")
    trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=test_dataloader)

    print("Start val")
    trainer.validate(model, test_dataloader)

    print("Skip XAI")

    print("XAI Eval")
    model.eval()
    all = []
    for images, targets in tqdm(test_dataloader):
        images, targets = images.to(device), targets.to(device)
        rs = rcap.batch_rcap(
            model.to(device), (images, targets),
            gradient_methods.guided_absolute_grad,
            {}
        )['overall_rcap']['RCAP']
        mlflow.log_metric('rcap', rs.mean())
        all.extend(rs)
    mlflow.log_metric('rcap', np.array(all).mean())

Using 16bit Automatic Mixed Precision (AMP)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type             | Params | Mode
----------------------------------------------------
0 | model   | EfficientNet     | 20.2 M | eval
1 | loss_fn | CrossEntropyLoss | 0      | eval
----------------------------------------------------
20.2 M    Trainable params
0         Non-trainable params
20.2 M    Total params
80.761    Total estimated model params size (MB)
0         Modules in train mode
715       Modules in eval mode


Start train


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


🏃 View run nimble-sponge-618 at: http://localhost:15001/#/experiments/0/runs/65a319bed27e4b1fb6f585cedbb003d3
🧪 View experiment at: http://localhost:15001/#/experiments/0


NameError: name 'exit' is not defined